# Imports

In [ ]:
from dotenv import load_dotenv
load_dotenv()
import numpy as np
import pandas as pd
import gc

from constants import AIRPORT_LIMIT_LIST, AIRLINE_LIMIT_LIST, KEEP_COLS
from loader import LoaderStorage

# S3

In [ ]:
storage = LoaderStorage(
  root="s3://data-mining/"
)

# the airlines are already filterd to the ones only that we use.
Source_folder = "data/raw/"
Destination_folder = "data/interim/"
output_file = "1_year_data_new.parquet"

# Loader

In [ ]:


# 2. Use a list to store dataframes temporarily
year_start = 14
year_end = 14
df_list = []
min_features_df_list = []
for j in range(year_start, year_end + 1):
    for i in range(1, 13):
        df_name = f"On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_20{j}_{i}.csv"
        file_path = Source_folder + df_name
        try:
            # 3. Only read the necessary columns (Saves massive RAM)
            # 4. Force C engine to avoid Arrow mask issues
            df_part = storage.read_csv(
                file_path,
                usecols=list(KEEP_COLS.keys()),
                dtype=KEEP_COLS,
                engine="c",
                low_memory=False
            )
            print(f"Processing Year: 20{j:02d}, Month: {i}")

            # 5. Filter immediately (Saves RAM for the final concat)
            mask = (
                df_part["Reporting_Airline"].isin(AIRLINE_LIMIT_LIST) &
                (df_part["Origin"].isin(AIRPORT_LIMIT_LIST) |
                df_part["Dest"].isin(AIRPORT_LIMIT_LIST))
            )
            df_part = df_part[mask]

            df_list.append(df_part)
        except FileNotFoundError:
            print(f"Skipping: {df_name} not found.")

# 6. Concatenate everything once at the end
if df_list:
    df_all = pd.concat(df_list, ignore_index=True)
else:
    df_all = pd.DataFrame(columns=list(KEEP_COLS.keys()))

# 7. Final cleanup
del df_list
gc.collect()

Drop anomaly flights that didn't make it to their destination despite not being cancelled or diverted

In [ ]:
df_all = df_all[~((df_all['ArrTime'].isna()) & (df_all['Cancelled'] == 0) & (df_all['Diverted'] == 0))]

In [ ]:
# colums with times of day
time_cols =["CRSDepTime","DepTime","CRSArrTime","ArrTime"]

cancelled_mask = (df_all['Cancelled'] == 1)
diverted_mask = (df_all['Diverted'] == 1)
for col in time_cols:
    missing_values = df_all[col].isna()
    df_all[col] = df_all[col].fillna(0).astype(int)
    # convert all 2400 values to 0000, because 2400 is not a valid time, but 0000 is
    df_all[col] = df_all[col].replace(2400, 0)
    df_all[col] = df_all[col].apply(lambda x: f"{x:04d}")
    df_all[col] = pd.to_datetime(df_all[col], format="%H%M", errors='coerce').dt.time
    # make all values NaT if the flight was canncelled
    df_all.loc[cancelled_mask & missing_values, col] = pd.NaT
df_all.loc[diverted_mask, 'ArrTime'] = pd.NaT

display(df_all[time_cols].head())

# Load airport information

In [ ]:
# load the airport dataset
airports = storage.read_csv('data/external/airports_with_runway_info.csv')
# only keep the iata_code and the Timezone
airports = airports[['iata_code','TZ']]
display(airports.head())

# get the unique values for TZ
print(airports['TZ'].unique())

# we join the aiports df_all to get the timezone of the departure and arrival airports
df_all = df_all.merge(airports, left_on='Origin', right_on='iata_code', how='left')
df_all = df_all.merge(airports, left_on='Dest', right_on='iata_code', how='left', suffixes=('_Origin', '_Dest'))
df_all = df_all.drop(columns=['iata_code_Origin', 'iata_code_Dest'])

In [ ]:
# step 1 is to get correct and reliable datetime objects in UTC and Local time for the departure and arrival times.
# we start with the departure time where we combine the filght date datetime with the daparture time timeobject to get a datetime object in local time
df_all['CRSDepDateTime'] = pd.to_datetime(df_all['FlightDate'] + ' ' + df_all['CRSDepTime'].astype(str), errors='coerce')
df_all['CRSDepDateTime_UTC'] = df_all.groupby('TZ_Origin')['CRSDepDateTime'].transform(
    lambda x: x.dt.tz_localize(x.name, ambiguous=True,nonexistent='shift_forward').dt.tz_convert('UTC')
)
# get a UTC_Flight date 
df_all['UTC_CRS_FlightDate'] = df_all['CRSDepDateTime_UTC'].dt.date
# we do the same for the arrival time
df_all['CRSArrDateTime'] = pd.to_datetime(df_all['FlightDate'] + ' ' + df_all['CRSArrTime'].astype(str), errors='coerce')
df_all['CRSArrDateTime_UTC'] = df_all.groupby('TZ_Dest')['CRSArrDateTime'].transform(
    lambda x: x.dt.tz_localize(x.name,ambiguous=True,nonexistent='shift_forward').dt.tz_convert('UTC')
)

# also convert the actual arrival and departure times to UTC time, by using the timezone of the departure and arrival airport
# check if the DepTime is smaller than the CRSDepTime, if it is, we add 1 day to the DepTime, because it means that the flight departed after midnight
mask = ((df_all['DepTime'] < df_all['CRSDepTime'] ) & (df_all['DepDelay'] > 0)).astype(int)

df_all['DepDateTime'] = pd.to_datetime(df_all['FlightDate'] + ' ' + df_all['DepTime'].astype(str), errors='coerce')
# add 1 day to the DepDateTime if the DepTime is smaller than the CRSDepTime, because it means that the flight departed after midnight
df_all['DepDateTime'] = df_all['DepDateTime'] + pd.to_timedelta(mask, unit='D')

df_all['DepDateTime_UTC'] = df_all.groupby('TZ_Origin')['DepDateTime'].transform(
    lambda x: x.dt.tz_localize(x.name, ambiguous=True,nonexistent='shift_forward').dt.tz_convert('UTC')
)

df_all['ArrDateTime'] = pd.to_datetime(df_all['FlightDate'] + ' ' + df_all['ArrTime'].astype(str), errors='coerce')
df_all['ArrDateTime_UTC'] = df_all.groupby('TZ_Dest')['ArrDateTime'].transform(
    lambda x: x.dt.tz_localize(x.name, ambiguous=True,nonexistent='shift_forward').dt.tz_convert('UTC')
)

# we check if the arrival time is before the departure time, if it is, we add 1 day to the arrival time
# we get a boolean mask where the arrival time is before the departure time
mask = (df_all['CRSArrDateTime_UTC'] < df_all['CRSDepDateTime_UTC']).astype(int)
df_all['CRSArrDateTime_UTC'] = df_all['CRSArrDateTime_UTC'] + pd.to_timedelta(mask, unit='D')
df_all['CRSArrDateTime'] = df_all['CRSArrDateTime'] + pd.to_timedelta(mask, unit='D')


mask = (df_all['ArrDateTime_UTC'] < df_all['DepDateTime_UTC']).astype(int)
df_all['ArrDateTime_UTC'] = df_all['ArrDateTime_UTC'] + pd.to_timedelta(mask, unit='D')
df_all['ArrDateTime'] = df_all['ArrDateTime'] + pd.to_timedelta(mask, unit='D')

display(df_all[['FlightDate','CRSDepTime', 'CRSDepDateTime', 'CRSDepDateTime_UTC', 'CRSArrTime', 'CRSArrDateTime_UTC']].head())

In [ ]:
# make a temporary Collumn for k hours before departure
hours_before = 2
df_all["Information_time_UTC"] = df_all["CRSDepDateTime_UTC"] - pd.to_timedelta(hours_before, unit='h')
# remove the timezone information, as the weather data does not have timezone information
df_all["Information_time_UTC"] = df_all["Information_time_UTC"].dt.tz_localize(None)
# get the last hourmark for every flight before the 2h before takeoff threashold.
# -> Needed for some features, where we only have information about the clean hour (like the weather data)
df_all['floor_informationtime_UTC'] = df_all['Information_time_UTC'].dt.floor('h')

# non UTC information time 
df_all["Information_time"] = df_all["CRSDepDateTime"] - pd.to_timedelta(hours_before, unit='h')
df_all["Information_time"] = df_all["Information_time"].dt.tz_localize(None) 
df_all['floor_informationtime'] = df_all['Information_time'].dt.floor('h') #for hour accutare Featuers

display(df_all)

In [ ]:
# save the cleaned data to a new parquet file (write directly to S3 with storage options)
target = f"s3://data-mining/{Destination_folder}{output_file}"
df_all.to_parquet(target, index=False, storage_options=storage.storage_options)